## Continuous Delivery with GitHub Actions

In this notebook, we look at how Continuous Delivery (CD) has been set up on GitHub for `Flixtube`.

**Note that you must have completed the previous notebook (01_github_monorepo) before working through this notebook.**

---

## Examine the GitHub Workflows

- Visit `https://github.com/<YourGitHubAccount>/monorepo/actions`
  - Notice the workflows in the left margin.
    - Click `Show more workflows...` to show all workflows.
    - Notice the `Integrate <Microservice>` workflows in the left margin (this is the automated CI part of the CI/CD  pipepline).
    - Notice the `Deploy <Microservice>` workflows in the left margin (this is the manual CD part of the CI/CD pipepline).

### The Continuous Delivery (CD) Workflow for `Flixtube.Web`

- Let's examine the `Deploy Flixtube.Web` workflow.
  - Click on `Deploy Flixtube.Web` in the left margin.
  - Then click the `web-cd.yaml` link in the top left, which will take you to the `Code`.
  - Notice in the left margin that this file is located in the subfolder `.github/workflows`.
    - **All GitHub workflows have to be located in this subfolder**.
  - In the main area, the contents of the file is displayed.

- Let's open the file in VSCode instead.
  - Right-click the file `.github/workflows/web-cd.yaml`, choose `Open to the side` and look at the various commads in the file.
  - `name` gives the workflow a name (displayed under GitHub Actions).
  - `on` controls if and when the workflow is triggered.
    - `workflow_dispatch` means this workflow can be triggered manually from GitHub's GUI.
    - Notice that `push` or `pull_request` is not defined for this workflow, so the workflow can only be triggered manually.
  - `jobs` is used to define one or more jobs (a workflow is made up of one or more jobs).
    - `deploy` is the name of the one and only job defined in this workflow.
      - `runs-on` determines what image is used to execute the *runner* on.
        - The current setting `ubuntu-latest` means an Ubuntu-based container will be used to run all commands in.
      - `env` is used to define environment variables available to all `steps` in the job.
        - In this case numerous environment variables are defined.
        - Notice many environment variables' values are set using *GitHub variables*.
          - A *GitHub variable's* value is obtained by placing it between double curly braces.
          - `secrets.<NAME>` is either a `SECRET` or a `VARIABLE` defined under `Settings -> Secrets and variables -> Actions` on GitHub.
          - `github.sha` will return the Secure Hasing Algorithm (SHA) hash for the latest Git commit that trigged the workflow.
      - `steps` contains a list of steps (tasks) that will be executed sequentailly.
        - `name` is used to name a step.
        - `uses` executes a pre-defined action.
          - Most pre-defined actions can be found here: https://github.com/actions
          - `with` configures a pre-defined action with name-value pairs specific to the action.
        - `run` is used to run a command in the container (based on `ubuntu-latest` in this case).
      - The current steps do the following.
        - `uses: actions/checkout@v3` (https://github.com/actions/checkout) checks out the GitHub repository to the *runner's* container (i.e. copies the GitHub repository to the container based on `ubuntu-latest`).
        - `run: chmod +x ./scripts/cicd/build-image.sh && ./scripts/cicd/build-image.sh` executes the given command in the *runner's* container.
          - This will make the script `build-image.sh` executable.
          - Then the script `build-image.sh` is executed in the *runner's* container.
          - If you look at the file `scripts/cicd/build-image.sh` you'll see that:
            - It uses four of the environment variables set above under `env`.
            -  It runs `docker build` to build the `Flixtube.Web` image.
        - `run: chmod +x ./scripts/cicd/push-image.sh && ./scripts/cicd/push-image.sh` executes the given command in the *runner's* container.
          - This will make the script `push-image.sh` executable.
          - Then the script `push-image.sh` is executed in the *runner's* container.
          - If you look at the file `scripts/cicd/push-image.sh` you'll see that:
            - It uses five of the environment variables set above under `env`.
            - It runs `docker login` to login to the Azure Container Registry. 
            -  It runs `docker push` to push the `Flixtube.Web` image to the Azure Container Registry.
        - `uses: tale/kubectl-action@v1` (https://github.com/tale/kubectl-action) installs `kubectl` to  the *runner's* conatiner and connects to the Azure Kubernetes Services (AKS) cluster.
          - In this case, it `kubectl` version `1.24.2`.
          - Also notice that is uses a specific kube config from a GitHub Actions SECRET.
        - `run: chmod +x ./scripts/cicd/deploy.sh && ./scripts/cicd/deploy.sh` executes the given command in the *runner's* container.
          - This will make the script `deploy.sh` executable.
          - Then the script `deploy.sh` is executed in the *runner's* container.
          - If you look at the file `scripts/cicd/deploy.sh` you'll see that:
            - It uses six of the environment variables set above under `env`. 
            - It runs `kubectl apply` using the microservices Kubenetes YAML manifest file to deploy the microservice to the  Azure Kubernetes Services (AKS) cluster.

- **Note that the other CD workflow files have a similar structure.**

---

## Bring up the Infrastructure with Terraform

### Change the Terraform variable `app_name` in `variables.tf`

- Open the file `variables.tf` in the subfolder `monorepo/terraform` on your local computer.
- Give your Terraform variable `app_name` a unique value.

  ```bash
  variable "app_name" {
    default = "flixtube2025g00" # <--- change this value to something unique
  }
  ```

### Get your `subscription_id`

- Your `subscription_id` needs to be set in the file `providers.tf` in the folder `monorepo/terraform`.

  ```bash
  provider "azurerm" {
    features {}
    subscription_id = var.subscription_id  # <--- this needs to be set to your Azure Subscription ID
  }
  ```
- The best way to do this is via an environment variable
  `TF_VAR_subscription_id`.

- To get your Azure Subscription ID, execute the cell below (we will use the Subscription ID in the next cell).

In [ ]:
!az account show --query id --output tsv

### Create the Infrastructure with Terraform Init and Terraform Apply

- Create the Azure infrastructure from the `terraform` folder inte `monorepo` subfolder.
  - Open a new terminal and change into the `monorepo/terraform` folder.
  - Run the command below (**replace `<SubscriptionId>` with your Azure Subscription ID from the cell above**):
    - **Windows**: `set TF_VAR_subscription_id=<SubscriptionId>`
    - **Linux/Mac**: `export TF_VAR_subscription_id=<SubscriptionId>`
  - Run the command `terraform init`
  - Run the command `terraform apply -auto-approve`

You should see something simlar to the below ...

```bash
Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following        
symbols:
  + create

Terraform will perform the following actions:

  # azurerm_container_registry.main will be created
  + resource "azurerm_container_registry" "main" {
      + admin_enabled                 = true
      + admin_password                = (sensitive value)
      + admin_username                = (known after apply)
      + encryption                    = (known after apply)
      + export_policy_enabled         = true
      + id                            = (known after apply)
      + location                      = "westeurope"
      + login_server                  = (known after apply)
      + name                          = "flixtube2025g00"
      + network_rule_bypass_option    = "AzureServices"
      + network_rule_set              = (known after apply)
      + public_network_access_enabled = true
      + resource_group_name           = "flixtube2025g00"
      + sku                           = "Basic"
      + trust_policy_enabled          = false
      + zone_redundancy_enabled       = false
    }

  # azurerm_kubernetes_cluster.main will be created
  + resource "azurerm_kubernetes_cluster" "main" {
      + current_kubernetes_version          = (known after apply)
      + dns_prefix                          = "flixtube2025g00"
      + fqdn                                = (known after apply)
      + http_application_routing_zone_name  = (known after apply)
      + id                                  = (known after apply)
      + kube_admin_config                   = (sensitive value)
      + kube_admin_config_raw               = (sensitive value)
      + kube_config                         = (sensitive value)
      + kube_config_raw                     = (sensitive value)
      + kubernetes_version                  = "1.30.6"
      + location                            = "westeurope"
      + name                                = "flixtube2025g00"
      + node_os_upgrade_channel             = "NodeImage"
      + node_resource_group                 = (known after apply)
      + node_resource_group_id              = (known after apply)
      + oidc_issuer_url                     = (known after apply)
      + portal_fqdn                         = (known after apply)
      + private_cluster_enabled             = false
      + private_cluster_public_fqdn_enabled = false
      + private_dns_zone_id                 = (known after apply)
      + private_fqdn                        = (known after apply)
      + resource_group_name                 = "flixtube2025g00"
      + role_based_access_control_enabled   = true
      + run_command_enabled                 = true
      + sku_tier                            = "Free"
      + support_plan                        = "KubernetesOfficial"
      + workload_identity_enabled           = false

      + auto_scaler_profile (known after apply)

      + default_node_pool {
          + kubelet_disk_type    = (known after apply)
          + max_pods             = (known after apply)
          + name                 = "default"
          + node_count           = 1
          + node_labels          = (known after apply)
          + orchestrator_version = (known after apply)
          + os_disk_size_gb      = (known after apply)
          + os_disk_type         = "Managed"
          + os_sku               = (known after apply)
          + scale_down_mode      = "Delete"
          + type                 = "VirtualMachineScaleSets"
          + ultra_ssd_enabled    = false
          + vm_size              = "Standard_D2S_V3"
          + workload_runtime     = (known after apply)
        }

      + identity {
          + principal_id = (known after apply)
          + tenant_id    = (known after apply)
          + type         = "SystemAssigned"
        }

      + kubelet_identity (known after apply)

      + network_profile (known after apply)

      + windows_profile (known after apply)
    }

  # azurerm_network_watcher.networkwatcher will be created
  + resource "azurerm_network_watcher" "networkwatcher" {
      + id                  = (known after apply)
      + location            = "westeurope"
      + name                = "NetworkWatcher_westeurope"
      + resource_group_name = "NetworkWatcherRG"
    }

  # azurerm_resource_group.main will be created
  + resource "azurerm_resource_group" "main" {
      + id       = (known after apply)
      + location = "westeurope"
      + name     = "flixtube2025g00"
    }

  # azurerm_resource_group.networkwatcher will be created
  + resource "azurerm_resource_group" "networkwatcher" {
      + id       = (known after apply)
      + location = "westeurope"
      + name     = "NetworkWatcherRG"
    }

  # azurerm_role_assignment.main will be created
  + resource "azurerm_role_assignment" "main" {
      + condition_version                = (known after apply)
      + id                               = (known after apply)
      + name                             = (known after apply)
      + principal_id                     = (known after apply)
      + principal_type                   = (known after apply)
      + role_definition_id               = (known after apply)
      + role_definition_name             = "AcrPull"
      + scope                            = (known after apply)
      + skip_service_principal_aad_check = true
    }

  # azurerm_storage_account.main will be created
  + resource "azurerm_storage_account" "main" {
      + access_tier                        = (known after apply)
      + account_kind                       = "StorageV2"
      + account_replication_type           = "LRS"
      + account_tier                       = "Standard"
      + allow_nested_items_to_be_public    = true
      + cross_tenant_replication_enabled   = false
      + default_to_oauth_authentication    = false
      + dns_endpoint_type                  = "Standard"
      + https_traffic_only_enabled         = true
      + id                                 = (known after apply)
      + infrastructure_encryption_enabled  = false
      + is_hns_enabled                     = false
      + large_file_share_enabled           = (known after apply)
      + local_user_enabled                 = true
      + location                           = "westeurope"
      + min_tls_version                    = "TLS1_2"
      + name                               = "flixtube2025g00"
      + nfsv3_enabled                      = false
      + primary_access_key                 = (sensitive value)
      + primary_blob_connection_string     = (sensitive value)
      + primary_blob_endpoint              = (known after apply)
      + primary_blob_host                  = (known after apply)
      + primary_blob_internet_endpoint     = (known after apply)
      + primary_blob_internet_host         = (known after apply)
      + primary_blob_microsoft_endpoint    = (known after apply)
      + primary_blob_microsoft_host        = (known after apply)
      + primary_connection_string          = (sensitive value)
      + primary_dfs_endpoint               = (known after apply)
      + primary_dfs_host                   = (known after apply)
      + primary_dfs_internet_endpoint      = (known after apply)
      + primary_dfs_internet_host          = (known after apply)
      + primary_dfs_microsoft_endpoint     = (known after apply)
      + primary_dfs_microsoft_host         = (known after apply)
      + primary_file_endpoint              = (known after apply)
      + primary_file_host                  = (known after apply)
      + primary_file_internet_endpoint     = (known after apply)
      + primary_file_internet_host         = (known after apply)
      + primary_file_microsoft_endpoint    = (known after apply)
      + primary_file_microsoft_host        = (known after apply)
      + primary_location                   = (known after apply)
      + primary_queue_endpoint             = (known after apply)
      + primary_queue_host                 = (known after apply)
      + primary_queue_microsoft_endpoint   = (known after apply)
      + primary_queue_microsoft_host       = (known after apply)
      + primary_table_endpoint             = (known after apply)
      + primary_table_host                 = (known after apply)
      + primary_table_microsoft_endpoint   = (known after apply)
      + primary_table_microsoft_host       = (known after apply)
      + primary_web_endpoint               = (known after apply)
      + primary_web_host                   = (known after apply)
      + primary_web_internet_endpoint      = (known after apply)
      + primary_web_internet_host          = (known after apply)
      + primary_web_microsoft_endpoint     = (known after apply)
      + primary_web_microsoft_host         = (known after apply)
      + public_network_access_enabled      = true
      + queue_encryption_key_type          = "Service"
      + resource_group_name                = "flixtube2025g00"
      + secondary_access_key               = (sensitive value)
      + secondary_blob_connection_string   = (sensitive value)
      + secondary_blob_endpoint            = (known after apply)
      + secondary_blob_host                = (known after apply)
      + secondary_blob_internet_endpoint   = (known after apply)
      + secondary_blob_internet_host       = (known after apply)
      + secondary_blob_microsoft_endpoint  = (known after apply)
      + secondary_blob_microsoft_host      = (known after apply)
      + secondary_connection_string        = (sensitive value)
      + secondary_dfs_endpoint             = (known after apply)
      + secondary_dfs_host                 = (known after apply)
      + secondary_dfs_internet_endpoint    = (known after apply)
      + secondary_dfs_internet_host        = (known after apply)
      + secondary_dfs_microsoft_endpoint   = (known after apply)
      + secondary_dfs_microsoft_host       = (known after apply)
      + secondary_file_endpoint            = (known after apply)
      + secondary_file_host                = (known after apply)
      + secondary_file_internet_endpoint   = (known after apply)
      + secondary_file_internet_host       = (known after apply)
      + secondary_file_microsoft_endpoint  = (known after apply)
      + secondary_file_microsoft_host      = (known after apply)
      + secondary_location                 = (known after apply)
      + secondary_queue_endpoint           = (known after apply)
      + secondary_queue_host               = (known after apply)
      + secondary_queue_microsoft_endpoint = (known after apply)
      + secondary_queue_microsoft_host     = (known after apply)
      + secondary_table_endpoint           = (known after apply)
      + secondary_table_host               = (known after apply)
      + secondary_table_microsoft_endpoint = (known after apply)
      + secondary_table_microsoft_host     = (known after apply)
      + secondary_web_endpoint             = (known after apply)
      + secondary_web_host                 = (known after apply)
      + secondary_web_internet_endpoint    = (known after apply)
      + secondary_web_internet_host        = (known after apply)
      + secondary_web_microsoft_endpoint   = (known after apply)
      + secondary_web_microsoft_host       = (known after apply)
      + sftp_enabled                       = false
      + shared_access_key_enabled          = true
      + table_encryption_key_type          = "Service"

      + blob_properties (known after apply)

      + network_rules (known after apply)

      + queue_properties (known after apply)

      + routing (known after apply)

      + share_properties (known after apply)

      + static_website (known after apply)
    }

  # azurerm_storage_container.main will be created
  + resource "azurerm_storage_container" "main" {
      + container_access_type             = "private"
      + default_encryption_scope          = (known after apply)
      + encryption_scope_override_enabled = true
      + has_immutability_policy           = (known after apply)
      + has_legal_hold                    = (known after apply)
      + id                                = (known after apply)
      + metadata                          = (known after apply)
      + name                              = "videos"
      + resource_manager_id               = (known after apply)
      + storage_account_id                = (known after apply)
    }

Plan: 8 to add, 0 to change, 0 to destroy.

Changes to Outputs:
  + AZURE_CONTAINER_REGISTRY_HOSTNAME = (known after apply)
  + AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value)
  + AZURE_CONTAINER_REGISTRY_USERNAME = (known after apply)
  + AZURE_STORAGE_ACCOUNT_KEY         = (sensitive value)
  + AZURE_STORAGE_ACCOUNT_NAME        = "flixtube2025g00"
azurerm_resource_group.networkwatcher: Creating...
azurerm_resource_group.main: Creating...
azurerm_resource_group.networkwatcher: Still creating... [10s elapsed]
azurerm_resource_group.main: Still creating... [10s elapsed]
azurerm_resource_group.main: Creation complete after 12s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_container_registry.main: Creating...
azurerm_storage_account.main: Creating...
azurerm_kubernetes_cluster.main: Creating...
azurerm_resource_group.networkwatcher: Creation complete after 12s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG]
azurerm_network_watcher.networkwatcher: Creating...
azurerm_network_watcher.networkwatcher: Creation complete after 2s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope]
azurerm_container_registry.main: Still creating... [10s elapsed]
azurerm_storage_account.main: Still creating... [10s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [10s elapsed]
azurerm_container_registry.main: Still creating... [20s elapsed]
azurerm_storage_account.main: Still creating... [20s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [20s elapsed]
azurerm_container_registry.main: Creation complete after 25s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_storage_account.main: Still creating... [30s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [30s elapsed]
azurerm_storage_account.main: Still creating... [40s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [40s elapsed]
azurerm_storage_account.main: Still creating... [50s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [50s elapsed]
azurerm_storage_account.main: Still creating... [1m0s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m0s elapsed]
azurerm_storage_account.main: Still creating... [1m10s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m10s elapsed]
azurerm_storage_account.main: Creation complete after 1m13s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00]
azurerm_storage_container.main: Creating...
azurerm_storage_container.main: Creation complete after 1s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos]
azurerm_kubernetes_cluster.main: Still creating... [1m20s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m30s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m40s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [1m50s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m0s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m10s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m20s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m30s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m40s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [2m50s elapsed]
azurerm_kubernetes_cluster.main: Still creating... [3m0s elapsed]
azurerm_kubernetes_cluster.main: Creation complete after 3m8s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00]
azurerm_role_assignment.main: Creating...
azurerm_role_assignment.main: Still creating... [10s elapsed]
azurerm_role_assignment.main: Still creating... [20s elapsed]
azurerm_role_assignment.main: Creation complete after 23s [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/c9446640-e167-7de0-6b81-94f716f3e620]

Apply complete! Resources: 8 added, 0 changed, 0 destroyed.

Outputs:

AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io"
AZURE_CONTAINER_REGISTRY_PASSWORD = <sensitive>
AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00"
AZURE_STORAGE_ACCOUNT_KEY = <sensitive>
AZURE_STORAGE_ACCOUNT_NAME = "flixtube2025g00"
```

---

## Get Azure Resource Credentials

### Get the Credentials via Terraform

- Run the cell below to get the credentials for the Azure resources created by Terraform
  - The first command returns all credentials in JSON format.
  - The other commands:
    - Extract the credentials from the JSON document.
    - Store them in python variables so we can reuse them in other cells in this notebook.

In [ ]:
!terraform -chdir=monorepo/terraform output --json

import json

RESULT=!terraform -chdir=monorepo/terraform output --json
json_dict=json.loads("\n ".join(RESULT).replace('\n', ''))

AZURE_CONTAINER_REGISTRY_HOSTNAME = json_dict['AZURE_CONTAINER_REGISTRY_HOSTNAME']['value']
AZURE_CONTAINER_REGISTRY_USERNAME = json_dict['AZURE_CONTAINER_REGISTRY_USERNAME']['value']
AZURE_CONTAINER_REGISTRY_PASSWORD = json_dict['AZURE_CONTAINER_REGISTRY_PASSWORD']['value']
AZURE_STORAGE_ACCOUNT_NAME = json_dict['AZURE_STORAGE_ACCOUNT_NAME']['value']
AZURE_STORAGE_ACCOUNT_KEY = json_dict['AZURE_STORAGE_ACCOUNT_KEY']['value']
APP_NAME = AZURE_STORAGE_ACCOUNT_NAME

print(f"AZURE_CONTAINER_REGISTRY_HOSTNAME: {AZURE_CONTAINER_REGISTRY_HOSTNAME}")
print(f"AZURE_CONTAINER_REGISTRY_USERNAME: {AZURE_CONTAINER_REGISTRY_USERNAME}")
print(f"AZURE_CONTAINER_REGISTRY_PASSWORD: {AZURE_CONTAINER_REGISTRY_PASSWORD}")
print(f"AZURE_STORAGE_ACCOUNT_NAME: {AZURE_STORAGE_ACCOUNT_NAME}")
print(f"AZURE_STORAGE_ACCOUNT_KEY: {AZURE_STORAGE_ACCOUNT_KEY}")
print(f"APP_NAME: {APP_NAME}")

### Get the Credentials via Azure CLI `az`

- We can get the same information via the Azure CLI tool `az`.

In [ ]:
AZURE_CONTAINER_REGISTRY_HOSTNAME=!az acr show -n {APP_NAME} --query loginServer -o tsv
AZURE_CONTAINER_REGISTRY_HOSTNAME=AZURE_CONTAINER_REGISTRY_HOSTNAME[0]
AZURE_CONTAINER_REGISTRY_USERNAME=!az acr credential show -n {APP_NAME} --query username -o tsv
AZURE_CONTAINER_REGISTRY_USERNAME=AZURE_CONTAINER_REGISTRY_USERNAME[0]
AZURE_CONTAINER_REGISTRY_PASSWORD=!az acr credential show -n {APP_NAME} --query passwords[0].value -o tsv
AZURE_CONTAINER_REGISTRY_PASSWORD=AZURE_CONTAINER_REGISTRY_PASSWORD[0]
AZURE_STORAGE_ACCOUNT_NAME=!az storage account list --query [0].name -o tsv
AZURE_STORAGE_ACCOUNT_NAME=AZURE_STORAGE_ACCOUNT_NAME[0]
AZURE_STORAGE_ACCOUNT_KEY=!az storage account keys list --account-name {APP_NAME} --resource-group {APP_NAME} --query [0].value -o tsv
AZURE_STORAGE_ACCOUNT_KEY=AZURE_STORAGE_ACCOUNT_KEY[0]

print(f"AZURE_CONTAINER_REGISTRY_HOSTNAME: {AZURE_CONTAINER_REGISTRY_HOSTNAME}")
print(f"AZURE_CONTAINER_REGISTRY_USERNAME: {AZURE_CONTAINER_REGISTRY_USERNAME}")
print(f"AZURE_CONTAINER_REGISTRY_PASSWORD: {AZURE_CONTAINER_REGISTRY_PASSWORD}")
print(f"AZURE_STORAGE_ACCOUNT_NAME: {AZURE_STORAGE_ACCOUNT_NAME}")
print(f"AZURE_STORAGE_ACCOUNT_KEY: {AZURE_STORAGE_ACCOUNT_KEY}")

---

## List Azure Resources

### Resources Created in Azure
- If you visit https://portal.azure.com/#browse/all you will see all the resources that have been created in Azure by Terraform.
- Let's use the Azure CLI `az` to list various resources below.  

### List Azure Resource Groups

- We see that three Azure Resource Groups were created from the `terraform` folder.
  - `NetworkWatcherRG`, `flixtube2025g00` and `MC_flixtube2025g00_flixtube2025g00_westeurope`

In [3]:
!az group list -o table

Name                                           Location    Status
---------------------------------------------  ----------  ---------
flixtube2025g00                                westeurope  Succeeded
NetworkWatcherRG                               westeurope  Succeeded
MC_flixtube2025g00_flixtube2025g00_westeurope  westeurope  Succeeded


### List Azure Kubernetes Services

- We see that an Azure Kubernetes Service (AKS) was created from the `terraform` folder.

In [4]:
!az aks list -o table

Name             Location    ResourceGroup    KubernetesVersion    CurrentKubernetesVersion    ProvisioningState    Fqdn
---------------  ----------  ---------------  -------------------  --------------------------  -------------------  -------------------------------------------------
flixtube2025g00  westeurope  flixtube2025g00  1.30.6               1.30.6                      Succeeded            flixtube2025g00-e0x6pz8q.hcp.westeurope.azmk8s.io


### List Azure Container Registries

- We see that an Azure Container Registry was created from the `terraform` folder.

In [5]:
!az acr list -o table

NAME             RESOURCE GROUP    LOCATION    SKU    LOGIN SERVER                CREATION DATE         ADMIN ENABLED
---------------  ----------------  ----------  -----  --------------------------  --------------------  ---------------
flixtube2025g00  flixtube2025g00   westeurope  Basic  flixtube2025g00.azurecr.io  2025-01-30T07:40:53Z  True


### List Azure Storage Accounts

- We see that an Azure Storage Account was created from the `terraform` folder.

In [6]:
!az storage account list -o table

AccessTier    AllowBlobPublicAccess    AllowCrossTenantReplication    AllowSharedKeyAccess    CreationTime                      DefaultToOAuthAuthentication    DnsEndpointType    EnableHttpsTrafficOnly    EnableNfsV3    IsHnsEnabled    IsLocalUserEnabled    IsSftpEnabled    Kind       Location    MinimumTlsVersion    Name             PrimaryLocation    ProvisioningState    PublicNetworkAccess    ResourceGroup    StatusOfPrimary
------------  -----------------------  -----------------------------  ----------------------  --------------------------------  ------------------------------  -----------------  ------------------------  -------------  --------------  --------------------  ---------------  ---------  ----------  -------------------  ---------------  -----------------  -------------------  ---------------------  ---------------  -----------------
Hot           True                     False                          True                    2025-01-30T07:40:50.618873+00:00  False 

### List Azure Storage Containers

- It also created a Storage Container called `videos` under the Azure Storage Account.

In [7]:
!az storage container list --account-name $AZURE_STORAGE_ACCOUNT_NAME --account-key $AZURE_STORAGE_ACCOUNT_KEY -o table

Name    Lease Status    Last Modified
------  --------------  -------------------------
videos                  2025-01-30T07:41:54+00:00


---

## Add Azure Kubernetes Cluster Info. to Local Kubectl Config File

- We need to fetch the configuration information from our Azure Kubernetes Service (AKS) and update our kubectl's config file.
- Here we are creating a backup of our current kubectl config file, before repalcing it will the AKS config file.
- Finally, we ensure we are using the AKS context when issuing kubectl commands.
- Also note that we are storing copy of the new config file in the file `aks-config`.

**Note**

- The command below attaches the Azure Kubernetes Cluster to the Azure Container Registry, so that the Azure Kubernetes Cluster can pull images from the Azure Container Registry.

  ```bash
  az aks update -n {APP_NAME} -g {APP_NAME} --attach-acr {APP_NAME} -o table
  ```

- But we don't have to do this here, since we do it in the Terraform file `kubernetes-cluster.tf`.

#### Run the cell below

In [8]:
# Use the commands below if on Windows
!copy %USERPROFILE%\.kube\config %USERPROFILE%\.kube\config.bak
!del %USERPROFILE%\.kube\config
!az aks get-credentials --name {APP_NAME} --resource-group {APP_NAME}
!az aks get-credentials --name {APP_NAME} --resource-group {APP_NAME} -f .\aks-config
!kubectl config current-context

# Use the commands below if on Linux/Mac
# !cp ~/.kube/config ~/.kube/config.bak
# !rm ~/.kube/config
# !az aks get-credentials --name {APP_NAME} --resource-group {APP_NAME}
# !az aks get-credentials --name {APP_NAME} --resource-group {APP_NAME} -f ./aks-config
# !kubectl config current-context

        1 file(s) copied.


flixtube2025g00


---

## Get Base64 Encoded Kubectl Config File

- We need one final piece of information to configure our GitHub Actions workflow:
  - A Base64-encoded version of our kubectl config file.
  - This is needed by the GitHub Actions CD workflows which will be issuing kubectl commands against the AKS cluster.
  - Let's store the Base64-encoded kubectl config file in a Python variable and print it out so we see what it looks like.
  - We will also store the Base64-encoded config file in the file `aks-config-base64`.

**Note**

- On Windows, before running the cell below:
  - Download Base64 from here: https://www.di-mgt.com.au/base64-for-windows.html
    - Here's a direct link: https://www.di-mgt.com.au/src/base64-1.2.0.zip
  - Place the binary `Base64.exe` in the folder `workshop5/02_Azure_and_Github_Actions`.

In [ ]:
!base64 .\aks-config > ./aks-config-base64
# !base64 ./aks-config > ./aks-config-base64 # use this on Linux/Mac

KUBE_CONFIG=!type .\aks-config-base64
KUBE_CONFIG=", ".join(KUBE_CONFIG).replace(', ', '')

print(f"KUBE_CONFIG:\n{KUBE_CONFIG}")

---

## Set GitHub Actions Secrets

### Setting GitHub Actions Secrets Manually

- We can add "secret" key-value pairs to our GitHub Repository, which can be accessed by our GitHub Actions Workflows.
- We do this so we don't hard-code sensitive information into our GitHub Actions Workflow files.
- You can do this manually in GitHub by:
  - Clicking on your `monorepo` Respository in GitHub.
  - Clicking on the `Settings` tab near the top of the web page.
  - Expanding the `Secrets and Values` combobox in the left margin on the web page and choosing `Actions`.
  - Clicking the button `New repository secret` under `Repository secrets`.
  - Entering a `Name` and a `Secret`, followed by clicking `Add secret`.
  - Repeat the previous step for all the necesary key-value pairs for the keys.
    - `CONTAINER_REGISTRY_LOGIN_SERVER`
    - `CONTAINER_REGISTRY_USERNAME`
    - `CONTAINER_REGISTRY_PASSWORD`
    - `STORAGE_ACCOUNT_NAME`
    - `STORAGE_ACCESS_KEY`
    - `KUBE_CONFIG`
  - The values (secrets) you need to enter for these keys (names) are repeated below.

In [ ]:
print(f"AZURE_CONTAINER_REGISTRY_HOSTNAME: {AZURE_CONTAINER_REGISTRY_HOSTNAME}")
print(f"AZURE_CONTAINER_REGISTRY_USERNAME: {AZURE_CONTAINER_REGISTRY_USERNAME}")
print(f"AZURE_CONTAINER_REGISTRY_PASSWORD: {AZURE_CONTAINER_REGISTRY_PASSWORD}")
print(f"AZURE_STORAGE_ACCOUNT_NAME: {AZURE_STORAGE_ACCOUNT_NAME}")
print(f"AZURE_STORAGE_ACCOUNT_KEY: {AZURE_STORAGE_ACCOUNT_KEY}")
print(f"KUBE_CONFIG: {KUBE_CONFIG}")

### Setting GitHub Actions Secrets via the GitHub CLI

- Alternatively, we can set the GitHub Actions Name-Secret pairs using the GitHub CLI below.
- Note, use secrets to hide values on GitHub, or variables to expose values.
  - `gh secret set <SECRET_NAME> --body <SECRET_VALUE>` sets a secret (value not visible on GitHub)
  - `gh variable set <VARIABLE_NAME> --body <VARIABLE_VALUE>` sets a variable (value visible on GitHub)
- The value for `KUBE_CONFIG` is too long to run in the notebook cell, so we read it in from the file `aks-config-base64` instead.

In [12]:
!cd monorepo && gh secret set AZURE_CONTAINER_REGISTRY_HOSTNAME --body {AZURE_CONTAINER_REGISTRY_HOSTNAME}
!cd monorepo && gh secret set AZURE_CONTAINER_REGISTRY_USERNAME --body {AZURE_CONTAINER_REGISTRY_USERNAME}
!cd monorepo && gh secret set AZURE_CONTAINER_REGISTRY_PASSWORD --body {AZURE_CONTAINER_REGISTRY_PASSWORD}
!cd monorepo && gh secret set AZURE_STORAGE_ACCOUNT_NAME --body {AZURE_STORAGE_ACCOUNT_NAME}
!cd monorepo && gh secret set AZURE_STORAGE_ACCOUNT_KEY --body {AZURE_STORAGE_ACCOUNT_KEY}
# !cd monorepo && gh secret set KUBE_CONFIG --body {KUBE_CONFIG}
!cd monorepo && gh secret set KUBE_CONFIG < ../aks-config-base64

!cd monorepo && gh secret list

AZURE_CONTAINER_REGISTRY_HOSTNAME	2025-01-30T07:52:18Z
AZURE_CONTAINER_REGISTRY_PASSWORD	2025-01-30T07:52:22Z
AZURE_CONTAINER_REGISTRY_USERNAME	2025-01-30T07:52:20Z
AZURE_STORAGE_ACCOUNT_KEY	2025-01-30T07:52:25Z
AZURE_STORAGE_ACCOUNT_NAME	2025-01-30T07:52:23Z
KUBE_CONFIG	2025-01-30T07:52:27Z


### Manually adding a GitHub Actions Secret

- Just as an example of how to manually add a GitHub Actions secret, here's how to add it for `KUBE_CONFIG`.
  - Visit `https://github.com/<YourGitHubAccount>/monorepo`
  - Click the `Settings` tab at the top of the web page.
  - In the left margin, under `Security`, expand `Secrets and variables`.
  - Click `Actions`.
  - In the main area, choose the `Secrets` tab.
  - Click the `New repository secret` button.
  - Under `NAME`, enter `KUBE_CONFIG`.
  - Under `SECRET`, copy and pase your Base64-encoded Kube Config file (run the cell below to get it).
  - Click the `Add secret` button.
  - You should now see six secrets under `Repository secrets`.

In [ ]:
print(f"{KUBE_CONFIG}")

## Deploy the Microservices to the AKS Cluster

### List the Continuous Delivery (CD) Workflows

- Let's use the GitHub CLI to list the GitHub Actions CD Workflows we have in our `monorepo` repository.

In [14]:
!cd monorepo && gh workflow list | findstr "Deploy"
# !cd monorepo && gh workflow list | grep -i "Deploy" # use this on Linux/Mac

Deploy Flixtube.AzureStorage	active	141100077
Deploy Flixtube.Gateway	active	141100079
Deploy Flixtube.History	active	141100081
Deploy Flixtube.Metadata	active	141100083
Deploy Minio	active	141100085
Deploy Flixtube.MinioStorage	active	141100086
Deploy RabbitMQ	active	141100088
Deploy SqlServer	active	141100089
Deploy Flixtube.VideoStreaming	active	141100090
Deploy Flixtube.VideoUpload	active	141100092
Deploy Flixtube.Web	active	141100094


### Deploy Backing Services and Microservices

- To deploy our `Flixtube` application to the Azure Kubernetes Service (AKS) cluster, we need to deploy:
  - The backing services:
    - `Deploy RabbitMQ`
    - `Deploy SqlServer`
    - Note, we don't want to `Deploy Minio` into production, since we are using an Azure Storage Container.
  - Our own microservices:
    - `Deploy Flixtube.AzureStorage`
    - `Deploy Flixtube.Metadata`
    - `Deploy Flixtube.History`
    - `Deploy Flixtube.VideoUpload`
    - `Deploy Flixtube.VideoStreaming`
    - `Deploy Flixtube.Gateway`
    - `Deploy Flixtube.Web`
    - Note, we don't want to `Deploy Flixtube.MinioStorage` into production, since we are using an Azure Storage Container.


### Deploy Via the GitHub User Interface

- Let's trigger one CD workflow using the GitHub Web UI.
  - Visit `https://github.com/<YourGitHubAccount>/monorepo`
  - Click the `Actions` tab at the top of the web page.
  - In the left margin, click `Show more workflows...` (to show all workflows).
  - Click `Deploy RabbitMQ`.
  - In the far right of the main area, click the `Run workflow` drop-down listbox.
  - Click the `Run workflow` button.
  - Click the `Actions` tab at the top of the web page.
  - Notice the workflow has been triggered.
  - Wait until the icon turns green, for a successful deployment. 

### Deploy the GitHub CLI

- Let's trigger the remaining CD workflows using the GitHub CLI by running the cell below.
- Note, the command we are using is `gh workflow run <WORKFLOW_NAME>`
  - `<WORKFLOW_NAME>` is the name of the workflow, e.g. `Deploy RabbitMQ`.
- Click on the `Actions` tab on GitHub and wait until all workflows have completed successfully. 

In [15]:
# Backing Services
# !cd monorepo && gh workflow run "Deploy RabbitMQ"
!cd monorepo && gh workflow run "Deploy SqlServer"

# Microservices
!cd monorepo && gh workflow run "Deploy Flixtube.AzureStorage"
!cd monorepo && gh workflow run "Deploy Flixtube.Metadata"
!cd monorepo && gh workflow run "Deploy Flixtube.History"
!cd monorepo && gh workflow run "Deploy Flixtube.VideoUpload"
!cd monorepo && gh workflow run "Deploy Flixtube.VideoStreaming"
!cd monorepo && gh workflow run "Deploy Flixtube.Gateway"
!cd monorepo && gh workflow run "Deploy Flixtube.Web"

---

## List Kuberenetes Resources

- Run the cell below to list the resources (pods, deployments and services) in the Kubernetes cluster.
- Notice all pods are up and running.
  - The pod named `db` is the `SqlServer` backing service.

In [17]:
!kubectl get pods,deployments,services

NAME                                   READY   STATUS    RESTARTS   AGE
pod/db-6687ff9b86-8mkms                1/1     Running   0          54s
pod/gateway-54cfd5fffb-4wcvl           1/1     Running   0          13s
pod/history-cc8d85886-7bj6w            1/1     Running   0          15s
pod/metadata-767d48ff47-xfqlr          1/1     Running   0          5s
pod/rabbit-d57559b69-rpdhl             1/1     Running   0          57s
pod/video-storage-556f45b6bb-7csgs     1/1     Running   0          16s
pod/video-streaming-5bd84ff6d6-lhqbc   1/1     Running   0          16s
pod/video-upload-d6fc49984-nnpxs       1/1     Running   0          19s

NAME                              READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/db                1/1     1            1           55s
deployment.apps/gateway           1/1     1            1           13s
deployment.apps/history           1/1     1            1           15s
deployment.apps/metadata          1/1     1            1           5

---

## Get the Public IP Address and Port number

- Notice the `web` and `gateway` `services` above are of type `LoadBalancer` with public IPAddresses (` EXTERNAL-IP`) and port numbers `PORT(S)`.
- Let's get the Services' public IP Addresses and Ports so we can access the kubernetes cluster over the internet.
- Let's store the Load Balancers' `EXTERNAL-IP` (public IP Address) and `PORT(S)` in Python variables so we can use them later in this notebook.

In [18]:
WEB_PUBLIC_IP=!kubectl get service web -o jsonpath='{.status.loadBalancer.ingress[0].ip}'
WEB_PUBLIC_IP=WEB_PUBLIC_IP[0].replace("'", "")
WEB_PUBLIC_PORT=!kubectl get service web -o jsonpath='{.spec.ports[0].port}'
WEB_PUBLIC_PORT=WEB_PUBLIC_PORT[0].replace("'", "")

GATEWAY_PUBLIC_IP=!kubectl get service gateway -o jsonpath='{.status.loadBalancer.ingress[0].ip}'
GATEWAY_PUBLIC_IP=GATEWAY_PUBLIC_IP[0].replace("'", "")
GATEWAY_PUBLIC_PORT=!kubectl get service gateway -o jsonpath='{.spec.ports[0].port}'
GATEWAY_PUBLIC_PORT=GATEWAY_PUBLIC_PORT[0].replace("'", "")

print(f"WEB_PUBLIC_IP: {WEB_PUBLIC_IP}")
print(f"WEB_PUBLIC_PORT: {WEB_PUBLIC_PORT}")

print(f"GATEWAY_PUBLIC_IP: {GATEWAY_PUBLIC_IP}")
print(f"GATEWAY_PUBLIC_PORT: {GATEWAY_PUBLIC_PORT}")

WEB_PUBLIC_IP: 4.175.211.68
WEB_PUBLIC_PORT: 4000
GATEWAY_PUBLIC_IP: 108.141.114.98
GATEWAY_PUBLIC_PORT: 4010


---

## Update the `Web` Microservice

- Since we didn't know the Gateway's public IP Address when we deployed the Web microservice, we need to update the Web microservice's environment variable `FLIXTUBE_PUBLIC_GATEWAY_HOST`.
- Run the cell below to update the environment variable.

In [19]:
!kubectl set env deployment/web FLIXTUBE_PUBLIC_GATEWAY_HOST={GATEWAY_PUBLIC_IP} -n default

deployment.apps/web env updated


---

## Test the Flixtube in the AKS Cluster

- Open a browser and enter the URL `http://WEB_PUBLIC_IP:WEB_PUBLIC_PORT`, where `WEB_PUBLIC_IP` and `WEB_PUBLIC_PORT` are the IP Address and Port for the Web microservice above.
  - Click the `Upload Video` button.
  - Click the `Choose File` button and select a video file.
    - You can upload the sample video file `SampleVideo_1280x720_1mb.mp4` in the folder `workshop5/02_Azure_and_Github_Actions`.
  -  Click `Upload`.
  -  Back on the Home Page, click the `Play` button to view the video.
  - Click the `Home` button when done.
  - Back on the Home Page, click the `Show Viewing History` button to show the viewing history.
  - Click the `Home` button when done.
  - Back on the Home Page, click the `Delete` button to delete the video.
  - Confirm the deletion by clicking the `Delete` button.
- Open a browser and enter the URL `http://GATEWAY_PUBLIC_IP:GATEWAY_PUBLIC_PORT/api/history`, where `GATEWAY_PUBLIC_IP` and `GATEWAY_PUBLIC_PORT` are the IP Address and Port for the gateway microservice above.
  - You should see a JSON document with the viewing history.

In [120]:
#!firefox http://{WEB_PUBLIC_IP}:{WEB_PUBLIC_PORT}
#!firefox http://{GATEWAY_PUBLIC_IP}:{GATEWAY_PUBLIC_PORT}/api/history

---

## Clean up

### Ensure the Kubectl Context is set to the `docker-desktop` Cluster

- Let's restore our local kubectl config file.
- The cell below will replace the existing kubectl config file with the backup we made before.
  - Followed by changing the context to `docker-desktop`.
    - If you aren't using Docker Desktop, use `kubectl config unset current-context` below instead of `kubectl config use-context docker-desktop`.

In [20]:
# Use the commands below if on Windows
!copy %USERPROFILE%\.kube\config.bak %USERPROFILE%\.kube\config
!kubectl config use-context docker-desktop
!kubectl config current-context

# Use the commands below if on Linux/Mac
# !cp ~/.kube/config.bak ~/.kube/config
# !kubectl config use-context docker-desktop
# !kubectl config current-context

        1 file(s) copied.
Switched to context "docker-desktop".
docker-desktop


## Destroy the Infrastructure with Terraform Destroy

- Let's destroy our infrastructure on Azure by running `terraform destroy` in the `terraform` folder in the `monorepo` folder.
  - Open a new terminal and change into the `workshop5/02_Azure_and_Github_Actions/monorepo/terraform` folder.
  - Run the command `az account show --query id --output tsv` to get your Azure Subscription ID.
  - Run the command below (**replace `<SubscriptionId>` with your Azure Subscription ID**):
    - **Windows**: `set TF_VAR_subscription_id=<SubscriptionId>`
    - **Linux/Mac**: `export TF_VAR_subscription_id=<SubscriptionId>`
  - Run the command `terraform destroy -auto-approve`

You should see something simlar to the below ...

```bash
azurerm_resource_group.networkwatcher: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG]
azurerm_resource_group.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_container_registry.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_storage_account.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00]
azurerm_kubernetes_cluster.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00]
azurerm_network_watcher.networkwatcher: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope]
azurerm_role_assignment.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/c9446640-e167-7de0-6b81-94f716f3e620]
azurerm_storage_container.main: Refreshing state... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos]

Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following
symbols:
  - destroy

Terraform will perform the following actions:

  # azurerm_container_registry.main will be destroyed
  - resource "azurerm_container_registry" "main" {
      - admin_enabled                 = true -> null
      - admin_password                = (sensitive value) -> null
      - admin_username                = "flixtube2025g00" -> null
      - anonymous_pull_enabled        = false -> null
      - data_endpoint_enabled         = false -> null
      - encryption                    = [] -> null
      - export_policy_enabled         = true -> null
      - id                            = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00" -> null
      - location                      = "westeurope" -> null
      - login_server                  = "flixtube2025g00.azurecr.io" -> null
      - name                          = "flixtube2025g00" -> null
      - network_rule_bypass_option    = "AzureServices" -> null
      - network_rule_set              = [] -> null
      - public_network_access_enabled = true -> null
      - quarantine_policy_enabled     = false -> null
      - resource_group_name           = "flixtube2025g00" -> null
      - retention_policy_in_days      = 0 -> null
      - sku                           = "Basic" -> null
      - tags                          = {} -> null
      - trust_policy_enabled          = false -> null
      - zone_redundancy_enabled       = false -> null
    }

  # azurerm_kubernetes_cluster.main will be destroyed
  - resource "azurerm_kubernetes_cluster" "main" {
      - cost_analysis_enabled               = false -> null
      - current_kubernetes_version          = "1.30.6" -> null
      - dns_prefix                          = "flixtube2025g00" -> null
      - fqdn                                = "flixtube2025g00-45rcumxl.hcp.westeurope.azmk8s.io" -> null
      - id                                  = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00" -> null
      - kube_admin_config                   = (sensitive value) -> null
      - kube_config                         = (sensitive value) -> null
      - kube_config_raw                     = (sensitive value) -> null
      - kubernetes_version                  = "1.30.6" -> null
      - local_account_disabled              = false -> null
      - location                            = "westeurope" -> null
      - name                                = "flixtube2025g00" -> null
      - node_os_upgrade_channel             = "NodeImage" -> null
      - node_resource_group                 = "MC_flixtube2025g00_flixtube2025g00_westeurope" -> null
      - node_resource_group_id              = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/MC_flixtube2025g00_flixtube2025g00_westeurope" -> null
      - oidc_issuer_enabled                 = false -> null
      - portal_fqdn                         = "flixtube2025g00-45rcumxl.portal.hcp.westeurope.azmk8s.io" -> null
      - private_cluster_enabled             = false -> null
      - private_cluster_public_fqdn_enabled = false -> null
      - resource_group_name                 = "flixtube2025g00" -> null
      - role_based_access_control_enabled   = true -> null
      - run_command_enabled                 = true -> null
      - sku_tier                            = "Free" -> null
      - support_plan                        = "KubernetesOfficial" -> null
      - tags                                = {} -> null
      - workload_identity_enabled           = false -> null
        # (8 unchanged attributes hidden)

      - default_node_pool {
          - auto_scaling_enabled          = false -> null
          - fips_enabled                  = false -> null
          - host_encryption_enabled       = false -> null
          - kubelet_disk_type             = "OS" -> null
          - max_count                     = 0 -> null
          - max_pods                      = 250 -> null
          - min_count                     = 0 -> null
          - name                          = "default" -> null
          - node_count                    = 1 -> null
          - node_labels                   = {} -> null
          - node_public_ip_enabled        = false -> null
          - only_critical_addons_enabled  = false -> null
          - orchestrator_version          = "1.30.6" -> null
          - os_disk_size_gb               = 128 -> null
          - os_disk_type                  = "Managed" -> null
          - os_sku                        = "Ubuntu" -> null
          - scale_down_mode               = "Delete" -> null
          - tags                          = {} -> null
          - type                          = "VirtualMachineScaleSets" -> null
          - ultra_ssd_enabled             = false -> null
          - vm_size                       = "Standard_D2S_V3" -> null
          - zones                         = [] -> null
            # (10 unchanged attributes hidden)

          - upgrade_settings {
              - drain_timeout_in_minutes      = 0 -> null
              - max_surge                     = "10%" -> null
              - node_soak_duration_in_minutes = 0 -> null
            }
        }

      - identity {
          - identity_ids = [] -> null
          - principal_id = "3e670504-1322-4775-a213-26a056e96b60" -> null
          - tenant_id    = "b907ed40-45b9-49d7-88d8-a4d6c026ede3" -> null
          - type         = "SystemAssigned" -> null
        }

      - kubelet_identity {
          - client_id                 = "ee8e122a-a9c4-4787-9cad-109a212e96eb" -> null
          - object_id                 = "c236097d-50d4-4358-8e18-dc73a2529894" -> null
          - user_assigned_identity_id = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/MC_flixtube2025g00_flixtube2025g00_westeurope/providers/Microsoft.ManagedIdentity/userAssignedIdentities/flixtube2025g00-agentpool" -> null
        }

      - network_profile {
          - dns_service_ip      = "10.0.0.10" -> null
          - ip_versions         = [
              - "IPv4",
            ] -> null
          - load_balancer_sku   = "standard" -> null
          - network_data_plane  = "azure" -> null
          - network_plugin      = "azure" -> null
          - network_plugin_mode = "overlay" -> null
          - outbound_type       = "loadBalancer" -> null
          - pod_cidr            = "10.244.0.0/16" -> null
          - pod_cidrs           = [
              - "10.244.0.0/16",
            ] -> null
          - service_cidr        = "10.0.0.0/16" -> null
          - service_cidrs       = [
              - "10.0.0.0/16",
            ] -> null
            # (2 unchanged attributes hidden)

          - load_balancer_profile {
              - backend_pool_type           = "NodeIPConfiguration" -> null
              - effective_outbound_ips      = [
                  - "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/MC_flixtube2025g00_flixtube2025g00_westeurope/providers/Microsoft.Network/publicIPAddresses/d546f7d4-dd75-494f-ba24-c097190b52bb",
                ] -> null
              - idle_timeout_in_minutes     = 0 -> null
              - managed_outbound_ip_count   = 1 -> null
              - managed_outbound_ipv6_count = 0 -> null
              - outbound_ip_address_ids     = [] -> null
              - outbound_ip_prefix_ids      = [] -> null
              - outbound_ports_allocated    = 0 -> null
            }
        }
    }

  # azurerm_network_watcher.networkwatcher will be destroyed
  - resource "azurerm_network_watcher" "networkwatcher" {
      - id                  = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope" -> null
      - location            = "westeurope" -> null
      - name                = "NetworkWatcher_westeurope" -> null
      - resource_group_name = "NetworkWatcherRG" -> null
      - tags                = {} -> null
    }

  # azurerm_resource_group.main will be destroyed
  - resource "azurerm_resource_group" "main" {
      - id         = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00" -> null
      - location   = "westeurope" -> null
      - name       = "flixtube2025g00" -> null
      - tags       = {} -> null
        # (1 unchanged attribute hidden)
    }

  # azurerm_resource_group.networkwatcher will be destroyed
  - resource "azurerm_resource_group" "networkwatcher" {
      - id         = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG" -> null
      - location   = "westeurope" -> null
      - name       = "NetworkWatcherRG" -> null
      - tags       = {} -> null
        # (1 unchanged attribute hidden)
    }

  # azurerm_role_assignment.main will be destroyed
  - resource "azurerm_role_assignment" "main" {
      - id                                     = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/c9446640-e167-7de0-6b81-94f716f3e620" -> null
      - name                                   = "c9446640-e167-7de0-6b81-94f716f3e620" -> null
      - principal_id                           = "c236097d-50d4-4358-8e18-dc73a2529894" -> null
      - principal_type                         = "ServicePrincipal" -> null
      - role_definition_id                     = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/providers/Microsoft.Authorization/roleDefinitions/7f951dda-4ed3-4680-a7ca-43fe172d538d" -> null
      - role_definition_name                   = "AcrPull" -> null
      - scope                                  = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00" -> null
      - skip_service_principal_aad_check       = true -> null
        # (4 unchanged attributes hidden)
    }

  # azurerm_storage_account.main will be destroyed
  - resource "azurerm_storage_account" "main" {
      - access_tier                        = "Hot" -> null
      - account_kind                       = "StorageV2" -> null
      - account_replication_type           = "LRS" -> null
      - account_tier                       = "Standard" -> null
      - allow_nested_items_to_be_public    = true -> null
      - cross_tenant_replication_enabled   = false -> null
      - default_to_oauth_authentication    = false -> null
      - dns_endpoint_type                  = "Standard" -> null
      - https_traffic_only_enabled         = true -> null
      - id                                 = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00" -> null
      - infrastructure_encryption_enabled  = false -> null
      - is_hns_enabled                     = false -> null
      - large_file_share_enabled           = false -> null
      - local_user_enabled                 = true -> null
      - location                           = "westeurope" -> null
      - min_tls_version                    = "TLS1_2" -> null
      - name                               = "flixtube2025g00" -> null
      - nfsv3_enabled                      = false -> null
      - primary_access_key                 = (sensitive value) -> null
      - primary_blob_connection_string     = (sensitive value) -> null
      - primary_blob_endpoint              = "https://flixtube2025g00.blob.core.windows.net/" -> null
      - primary_blob_host                  = "flixtube2025g00.blob.core.windows.net" -> null
      - primary_connection_string          = (sensitive value) -> null
      - primary_dfs_endpoint               = "https://flixtube2025g00.dfs.core.windows.net/" -> null
      - primary_dfs_host                   = "flixtube2025g00.dfs.core.windows.net" -> null
      - primary_file_endpoint              = "https://flixtube2025g00.file.core.windows.net/" -> null
      - primary_file_host                  = "flixtube2025g00.file.core.windows.net" -> null
      - primary_location                   = "westeurope" -> null
      - primary_queue_endpoint             = "https://flixtube2025g00.queue.core.windows.net/" -> null
      - primary_queue_host                 = "flixtube2025g00.queue.core.windows.net" -> null
      - primary_table_endpoint             = "https://flixtube2025g00.table.core.windows.net/" -> null
      - primary_table_host                 = "flixtube2025g00.table.core.windows.net" -> null
      - primary_web_endpoint               = "https://flixtube2025g00.z6.web.core.windows.net/" -> null
      - primary_web_host                   = "flixtube2025g00.z6.web.core.windows.net" -> null
      - public_network_access_enabled      = true -> null
      - queue_encryption_key_type          = "Service" -> null
      - resource_group_name                = "flixtube2025g00" -> null
      - secondary_access_key               = (sensitive value) -> null
      - secondary_connection_string        = (sensitive value) -> null
      - sftp_enabled                       = false -> null
      - shared_access_key_enabled          = true -> null
      - table_encryption_key_type          = "Service" -> null
      - tags                               = {} -> null
        # (56 unchanged attributes hidden)

      - blob_properties {
          - change_feed_enabled           = false -> null
          - change_feed_retention_in_days = 0 -> null
          - last_access_time_enabled      = false -> null
          - versioning_enabled            = false -> null
            # (1 unchanged attribute hidden)
        }

      - queue_properties {
          - hour_metrics {
              - enabled               = true -> null
              - include_apis          = true -> null
              - retention_policy_days = 7 -> null
              - version               = "1.0" -> null
            }
          - logging {
              - delete                = false -> null
              - read                  = false -> null
              - retention_policy_days = 0 -> null
              - version               = "1.0" -> null
              - write                 = false -> null
            }
          - minute_metrics {
              - enabled               = false -> null
              - include_apis          = false -> null
              - retention_policy_days = 0 -> null
              - version               = "1.0" -> null
            }
        }

      - share_properties {
          - retention_policy {
              - days = 7 -> null
            }
        }
    }

  # azurerm_storage_container.main will be destroyed
  - resource "azurerm_storage_container" "main" {
      - container_access_type             = "private" -> null
      - default_encryption_scope          = "$account-encryption-key" -> null
      - encryption_scope_override_enabled = true -> null
      - has_immutability_policy           = false -> null
      - has_legal_hold                    = false -> null
      - id                                = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos" -> null
      - metadata                          = {} -> null
      - name                              = "videos" -> null
      - resource_manager_id               = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos" -> null
      - storage_account_id                = "/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00" -> null
    }

Plan: 0 to add, 0 to change, 8 to destroy.

Changes to Outputs:
  - AZURE_CONTAINER_REGISTRY_HOSTNAME = "flixtube2025g00.azurecr.io" -> null
  - AZURE_CONTAINER_REGISTRY_PASSWORD = (sensitive value) -> null
  - AZURE_CONTAINER_REGISTRY_USERNAME = "flixtube2025g00" -> null
  - AZURE_STORAGE_ACCOUNT_KEY         = (sensitive value) -> null
  - AZURE_STORAGE_ACCOUNT_NAME        = "flixtube2025g00" -> null
azurerm_network_watcher.networkwatcher: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG/providers/Microsoft.Network/networkWatchers/NetworkWatcher_westeurope]
azurerm_storage_container.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00/blobServices/default/containers/videos]
azurerm_role_assignment.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00/providers/Microsoft.Authorization/roleAssignments/c9446640-e167-7de0-6b81-94f716f3e620]
azurerm_storage_container.main: Destruction complete after 1s
azurerm_storage_account.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.Storage/storageAccounts/flixtube2025g00]
azurerm_role_assignment.main: Destruction complete after 3s
azurerm_container_registry.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerRegistry/registries/flixtube2025g00]
azurerm_kubernetes_cluster.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00/providers/Microsoft.ContainerService/managedClusters/flixtube2025g00]
azurerm_storage_account.main: Destruction complete after 4s
azurerm_network_watcher.networkwatcher: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...workWatchers/NetworkWatcher_westeurope, 10s elapsed]
azurerm_network_watcher.networkwatcher: Destruction complete after 12s
azurerm_resource_group.networkwatcher: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/NetworkWatcherRG]
azurerm_container_registry.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...nerRegistry/registries/flixtube2025g00, 10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 10s elapsed]
azurerm_container_registry.main: Destruction complete after 16s
azurerm_resource_group.networkwatcher: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...575c23/resourceGroups/NetworkWatcherRG, 10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 20s elapsed]
azurerm_resource_group.networkwatcher: Destruction complete after 17s
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 30s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 40s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 50s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m0s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m20s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m30s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m40s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 1m50s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m0s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m20s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m30s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m40s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 2m50s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m0s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m10s elapsed]
azurerm_kubernetes_cluster.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...ervice/managedClusters/flixtube2025g00, 3m20s elapsed]
azurerm_kubernetes_cluster.main: Destruction complete after 3m26s
azurerm_resource_group.main: Destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45/resourceGroups/flixtube2025g00]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...7575c23/resourceGroups/flixtube2025g00, 10s elapsed]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...7575c23/resourceGroups/flixtube2025g00, 20s elapsed]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...7575c23/resourceGroups/flixtube2025g00, 30s elapsed]
azurerm_resource_group.main: Still destroying... [id=/subscriptions/b129u222-k872-3gg5-5jw6-5c9a93843b45-...7575c23/resourceGroups/flixtube2025g00, 40s elapsed]
azurerm_resource_group.main: Destruction complete after 48s

Destroy complete! Resources: 8 destroyed.
```

## Ensure Azure Resources Deleted

### List Azure Storage Accounts

- We see the Azure Storage Account has been destroyed.

In [21]:
!az storage account list -o table

### List Azure Container Registries

- We see the Azure Container Registry has been destroyed.

In [22]:
!az acr list -o table

### List Azure Kubernetes Services

- We see the Azure Kubernetes Service has been destroyed.

In [23]:
!az aks list -o table

### List Azure Resource Groups

- We see the Azure Resource Groups have been destroyed.

In [24]:
!az group list -o table

### Check Resources and Resourev Groups on the Azure Portal

- Visit https://portal.azure.com/#browse/all 
  - We see the Azure Resources have been destroyed.
- Visit https://portal.azure.com/#browse/resourcegroups 
  - We see the Azure Resource Groups have been destroyed.

## Delete `monorepo` Repository

- Now we can delete the `monorepo` GitHub repository and the `workshop5/02_Azure_and_Github_Actions/monorepo` folder to clean things up.

In [25]:
!gh repo delete monorepo --yes

!rmdir /S /Q monorepo
# !rm -rf monorepo # use this on Linux/Mac